In [ ]:
# Step 1: Import libraries
import pandas as pd
import numpy as np

# Export to DataBase
from sqlalchemy import create_engine, types

#  Physical activity data from Questionnaire

In [3]:
# Exporting the Dietary Nutrients intake day  data
df = pd.read_sas('..\data\Quest_data\PAQ_L.xpt', format="xport", encoding="utf-8")

In [4]:
# Step 1: Rename the columns

df = df.rename(columns={
    'SEQN': 'seqn_no',
    'PAD790Q': 'mod_LTPA_count', # moderate LTPA - leisure time physical activity
    'PAD790U': 'mod_LTPA_unit',
    'PAD800': 'mod_LTP_minutes',
    'PAD810Q': 'vig_LTP_count', # vigorous LTPA
    'PAD810U': 'vig_LTP_unit',
    'PAD820': 'vig_LTP_minutes',
    'PAD680': 'sed_minutes' # Sedantary activity minutes excluding sleep
})

# Step 2: Changing the data type of 'seqn_no' to int
df['seqn_no'] = df['seqn_no'].astype(int)

In [5]:
from dotenv import dotenv_values

config = dotenv_values()

# define variables for the login
pg_user = config['POSTGRES_USER']  # align the key label with your .env file !
pg_host = config['POSTGRES_HOST']
pg_port = config['POSTGRES_PORT']
pg_db = config['POSTGRES_DB']
pg_schema = config['POSTGRES_SCHEMA']
pg_pass = config['POSTGRES_PASS']

# Now building the URL with the values from the .env file
url = f'postgresql://{pg_user}:{pg_pass}@{pg_host}:{pg_port}/{pg_db}'

engine = create_engine(url, echo=False) 

In [6]:
df.columns

Index(['seqn_no', 'mod_LTPA_count', 'mod_LTPA_unit', 'mod_LTP_minutes',
       'vig_LTP_count', 'vig_LTP_unit', 'vig_LTP_minutes', 'sed_minutes'],
      dtype='object')

In [ ]:


# Fill NaN values in activity minutes and counts with 0, and units with an empty string for easier processing
df['mod_LTP_minutes'] = df['mod_LTP_minutes'].fillna(0)
df['mod_LTPA_unit'] = df['mod_LTPA_unit'].fillna('')
df['vig_LTP_minutes'] = df['vig_LTP_minutes'].fillna(0)
df['vig_LTP_unit'] = df['vig_LTP_unit'].fillna('')
df['mod_LTPA_count'] = df['mod_LTPA_count'].fillna(0)
df['vig_LTP_count'] = df['vig_LTP_count'].fillna(0)

# Initialize new columns for weekly minutes
df['moderate_activity_minutes_per_week'] = 0.0
df['vigorous_activity_minutes_per_week'] = 0.0

# Function to convert minutes based on unit to weekly minutes
def convert_to_weekly(row, minutes_col, unit_col, count_col):
    unit = str(row[unit_col]).strip().upper()
    minutes = row[minutes_col]
    count = row[count_col]

    if unit == 'W':
        return minutes * count
    elif unit == 'D':
        return minutes * count * 7
    elif unit == 'M':
        # Approximate conversion for monthly to weekly (assuming 4 weeks in a month)
        return minutes * count / 4
    else:
        return 0.0

# Apply the conversion function
df['moderate_activity_minutes_per_week'] = df.apply(lambda row: convert_to_weekly(row, 'mod_LTP_minutes', 'mod_LTPA_unit', 'mod_LTPA_count'), axis=1)
df['vigorous_activity_minutes_per_week'] = df.apply(lambda row: convert_to_weekly(row, 'vig_LTP_minutes', 'vig_LTP_unit', 'vig_LTP_count'), axis=1)

# Define activity classification function based on WHO guidelines for adults
# WHO recommends:
# At least 150–300 minutes of moderate-intensity aerobic physical activity OR
# At least 75–150 minutes of vigorous-intensity aerobic physical activity OR
# An equivalent combination across the week
# One minute of vigorous-intensity activity is equivalent to 2 minutes of moderate-intensity activity.

def classify_activity_level(row):
    moderate_minutes = row['moderate_activity_minutes_per_week']
    vigorous_minutes = row['vigorous_activity_minutes_per_week']

    # Calculate MET-minutes equivalent
    # 1 minute of vigorous activity is equivalent to 2 minutes of moderate activity
    equivalent_moderate_minutes = moderate_minutes + (vigorous_minutes * 2)

    # Classification based on WHO guidelines
    if equivalent_moderate_minutes >= 150:
        return 'Active'
    else:
        return 'Not Active'

# Apply the classification function
df['activity_level'] = df.apply(classify_activity_level, axis=1)

# Display a sample of the results with the new columns
print(df[['moderate_activity_minutes_per_week', 'vigorous_activity_minutes_per_week', 'activity_level', 'sed_minutes']].head(10))

# Display the value counts for activity_level
print("\nActivity Level Distribution:")
print(df['activity_level'].value_counts())

In [14]:
df.to_sql(name = 'Quest_physical_activity', con=engine, schema='capstone_group_3',if_exists='replace',index=False)

153

In [ ]:
# Describe sedentary minutes
print("Descriptive statistics for Sedentary Minutes:")
print(df['sed_minutes'].describe())

# Analyze sedentary minutes by activity level
print("\nAverage Sedentary Minutes by Activity Level:")
print(df.groupby('activity_level')['sed_minutes'].mean())


Average Sedentary Minutes by Activity Level:

As expected, there's a difference in sedentary minutes between the two activity groups:

Active individuals: Average sedentary minutes per day is about 390 minutes (6.5 hours).
Not Active individuals: Average sedentary minutes per day is about 511 minutes (8.5 hours).

# Sleep disorder data from Questionnaire

In [ ]:
# Exporting sleep disorder data from Questionaire
df_quest_sleep_disorders = pd.read_sas('..\data\Quest_data\SLQ_L.xpt', format="xport", encoding="utf-8")

In [ ]:
# Step 1: Rename the columns

df_quest_sleep_disorders = df_quest_sleep_disorders.rename(columns={
    'SEQN': 'seqn_no',
    'SLQ300': 'bedtime_weekday',
    'SLQ310': 'waketime_weekday',
    'SLD012': 'sleep_hrs_weekday', 
    'SLQ320': 'bedtime_weekend',
    'SLQ330': 'waketime_weekend',
    'SLD013': 'sleep_hrs_weekend'
})

# Step 2: Changing the data type of 'seqn_no' to int
df_quest_sleep_disorders['seqn_no'] = df_quest_sleep_disorders['seqn_no'].astype(int)

In [ ]:
df_quest_sleep_disorders

In [ ]:
# Equal Weight Average
df_quest_sleep_disorders['avg_sleep_hrs'] = df_quest_sleep_disorders[['sleep_hrs_weekday', 'sleep_hrs_weekend']].mean(axis=1)

In [ ]:
df_quest_sleep_disorders

In [ ]:
# Now building the URL with the values from the .env file
url = f'postgresql://{pg_user}:{pg_pass}@{pg_host}:{pg_port}/{pg_db}'

engine = create_engine(url, echo=False) 
df_quest_sleep_disorders.to_sql(name = 'Quest_sleep', con=engine, schema='capstone_group_3',if_exists='replace',index=False)